Напишете клас `SimpleBackup`, който да реализира следните функционалности:

- Създаване на копие на данни по подаден списък с файлове за копиране (`create`). Създаваме копие на всички файлове, но не и на самите директории.
- Възстановяване на копие на данни по подадено име на копие и локация, където да се възстанови (`restore`)
- Връщане на сортиран списък с всички копия на данни, които са направени (`show`)
- Изтриване на копие на данни по подадено име на копие (`delete`)

Име на копие дефинираме по следния начин: `backup_<timestamp>`, където `<timestamp>` е времето на създаване на копието във формат `YYYYMMDD_HHMMSS`.
(За форматирането на датата и часа може да използвате `strftime` от модула `time`.)

Списъка с файлове за копиране е в следния формат - на всеки ред е описан един файл с релативния му път.
Важно: пътищата на файловете са разделени с интервал, а не с `/`.

Копията трябва да се съхраняват в директорията `backups`, като всяко копие е в отделна директория.

Методите `create`, `restore`, `show` и `delete` на копие трябва да връщат `True` ако операцията е била успешна, и `False` в противен случай.

В директорията `w3_files` се намира `backups` (където трябва да бъдат създавани вашите копия), списъците с файлове за копиране - `backup_1.txt`, `backup_2.txt` и т.н., и директорията `sample_files`, в която се намират тестовите файлове, на които ще правим копия.

**Бележки:**

Изпълнявайте тетрадката/кода ви в директорията `workshop3`, за да може да работите с релативни пътища.

Независимо от структурата на входните ви данни, съдържанието на резервните копия трябва да е на едно ниво - т.е всички файлове за дадено копие трябва да се намират в една директория.

In [66]:
# Some helper code
import os


def get_folder_content(backup_dir: str) -> list[str]:
    result = []

    for content in os.listdir(backup_dir):
        with open(os.path.join(backup_dir, content), 'r') as f:
            result.append(f.read().strip())

    return result

In [78]:
# Write your code here:
import datetime
import re
import shutil


class SimpleBackup:
    def __init__(self):
        pass

    def create(self, file_path: str) -> bool:
        try:
            with open(file_path) as fp:
                real_paths = [self.__construct_path(line) for line in fp]
        except FileNotFoundError, OSError:
            return False

        backup_dir = os.path.join("w3_files", "backups", self.__generate_backup_name())

        try:
            os.makedirs(backup_dir, exist_ok=True)
        except OSError:
            return False

        try:
            for real_path in real_paths:
                shutil.copy2(real_path, backup_dir)
        except OSError:
            return False

        return True

    def restore(self, backup_name: str, target_dir: str) -> bool:
        backup_path = os.path.join("w3_files", "backups", backup_name)

        if not os.path.exists(backup_path) or not os.path.exists(target_dir):
            return False

        try:
            files = os.listdir(backup_path)
        except OSError:
            return False

        try:
            for file_name in files:
                file_path = os.path.join(backup_path, file_name)
                shutil.copy2(file_path, target_dir)
        except OSError:
            return False

        return True

    def show(self) -> list[str]:
        backups_path = os.path.join("w3_files", "backups")

        try:
            all_paths = os.listdir(backups_path)
        except OSError:
            return []

        exp = re.compile(r"backup_[0-9]{8}_[0-9]{6}")

        return sorted([path for path in all_paths if exp.match(path) is not None])

    def delete(self, backup_name: str):
        backup_path = os.path.join("w3_files", "backups", backup_name)

        if not os.path.exists(backup_path):
            return False

        try:
            shutil.rmtree(backup_path)
        except OSError:
            return False

        return True

    @staticmethod
    def __generate_backup_name() -> str:
        return datetime.datetime.now().strftime("backup_%Y%m%d_%H%M%S")


    @staticmethod
    def __construct_path(file_path: str) -> str:
        return os.path.join(*file_path.strip().split(" "))

In [79]:
backups = SimpleBackup()


# Arrange 
default_listing = backups.show()

# Act 
backup1_result = backups.create(os.path.join('w3_files', 'backup_1.txt'))
time.sleep(1)  # to ensure that the backups will have a different timestamp
backup2_result = backups.create(os.path.join('w3_files', 'backup_2.txt'))

listing1_result = backups.show()

backup1_path = listing1_result[-2]
backup2_path = listing1_result[-1]

backup1_content = get_folder_content(os.path.join('w3_files', 'backups', backup1_path))

In [80]:
!tree w3_files/

w3_files/
├── backup_1.txt
├── backup_2.txt
├── backup_3.txt
├── backup_4.txt
├── backups
│   ├── backup_20261123_204944
│   │   └── first.txt
│   ├── backup_20261123_204945
│   │   └── eight.txt
│   ├── backup_20261123_205044
│   │   └── first.txt
│   └── backup_20261123_205045
│       └── eight.txt
├── restored_backup_1
│   └── first.txt
└── sample_files
    ├── first.txt
    ├── fourth
    │   └── eight.txt
    ├── second.txt
    └── third
        └── fifth
            └── seventh.txt

11 directories, 13 files


In [84]:
import time
import shutil

backups = SimpleBackup()


# Arrange 
default_listing = backups.show()

# Act 
backup1_result = backups.create(os.path.join('w3_files', 'backup_1.txt'))
time.sleep(1)  # to ensure that the backups will have a different timestamp
backup2_result = backups.create(os.path.join('w3_files', 'backup_2.txt'))

listing1_result = backups.show()

backup1_path = listing1_result[-2]
backup2_path = listing1_result[-1]

backup1_content = get_folder_content(os.path.join('w3_files', 'backups', backup1_path))
backup2_content = get_folder_content(os.path.join('w3_files', 'backups', backup2_path))

time.sleep(1)  # to ensure that the backups will have a different timestamp

backup3_result = backups.create(os.path.join('w3_files', 'backup_3.txt'))
time.sleep(1)  # to ensure that the backups will have a different timestamp

backup4_result = backups.create(os.path.join('w3_files', 'backup_4.txt'))
time.sleep(1)  # to ensure that the backups will have a different timestamp

backup5_result = backups.create(os.path.join('w3_files', 'backup_5.txt'))
time.sleep(1)  # to ensure that the backups will have a different timestamp

listing2_result = backups.show()

backup3_path = listing2_result[-2]
backup4_path = listing2_result[-1]

backup3_content = get_folder_content(os.path.join('w3_files', 'backups', backup3_path))
backup4_content = get_folder_content(os.path.join('w3_files', 'backups', backup4_path))

backup6_result = backups.create(os.path.join('w3_files', 'backup_4.txt'))
listing3_result = sorted(backups.show())
time.sleep(1)  # to ensure that the backups will have a different timestamp

removal1_result = backups.delete(listing3_result[-1])
listing4_result = backups.show()


restore_results = []
restore_paths = [os.path.join('w3_files', f'restored_{i}') for i in range(1, 5)]
backups_to_restore = listing4_result[-4:]
restore_contents = []

for backup_to_restore, restore_path in zip(backups_to_restore, restore_paths):
    os.makedirs(restore_path)
    restore_results.append(backups.restore(backup_to_restore, restore_path))
    restore_contents.append(get_folder_content(restore_path))
    shutil.rmtree(restore_path)

os.makedirs(restore_paths[0])
restore5_result = backups.restore('backup_20261123_144719', restore_paths[0])
restore6_result = backups.restore('asd', restore_paths[0])
shutil.rmtree(restore_paths[0])

for backup in backups.show():
    backups.delete(backup)

listing5_result = backups.show()

# Tests

# Create
assert backup1_result == True
assert backup2_result == True
assert backup3_result == True
assert backup4_result == True
assert backup5_result == False, 'Config file for backup_5 does not exist'


assert set(backup1_content) == set(['123'])
assert set(backup2_content) == set(['456'])
assert set(backup3_content) == set(['456', '4567'])
assert set(backup4_content) == set(['123', '456', 'Атака, чичо !', '4567'])

# Show
assert len(listing1_result) == len(default_listing) + 2
assert len(listing2_result) == len(default_listing) + 4
assert len(listing3_result) == len(default_listing) + 5
assert len(listing4_result) == len(default_listing) + 4
assert len(listing5_result) == len(default_listing)
assert listing5_result == default_listing

# Restore
assert restore_results[0] == True
assert restore_results[1] == True
assert restore_results[2] == True
assert restore_results[3] == True
assert restore5_result == False, 'Restore of non-existing backup should fail'
assert restore6_result == False, 'Restore of backup with invalid name should fail'


assert set(restore_contents[0]) == set(['123'])
assert set(restore_contents[1]) == set(['456'])
assert set(restore_contents[2]) == set(['456', '4567'])
assert set(restore_contents[3]) == set(['123', '456', 'Атака, чичо !', '4567'])

# Delete
for backup in listing3_result:
    assert os.path.exists(os.path.join('w3_files', 'backups', backup)) == False


"✅ All OK! +4 points"

'✅ All OK! +4 points'